# 🏈 Fantasy Football Analytics: Models, Backtesting & Calibration

This notebook is the data-science engine behind a personal fantasy-football tool that
monitors five ESPN leagues and sends start/sit, waiver, and trade advice to a Discord bot.
The live tool needs private league access, so it lives in the repo. **What runs here is the
reproducible modeling and, crucially, the honest validation of it**, all on free public data.

What you'll see:
1. **Monte Carlo** floor/ceiling projections
2. **Value over replacement** (fair comparison across positions)
3. **"Heating up"** thresholds learned from history
4. **Backtesting** the recommendations across five seasons (the proof it works)
5. **Calibration** of the prediction ranges
6. **Game script** from live Vegas betting lines

> Full project, including the reports: **[github.com/jamesonjdavis-maker/fantasy-football-monitor](https://github.com/jamesonjdavis-maker/fantasy-football-monitor)**
> ([REPORT.md](https://github.com/jamesonjdavis-maker/fantasy-football-monitor/blob/main/REPORT.md) · [DETAILED_REPORT.md](https://github.com/jamesonjdavis-maker/fantasy-football-monitor/blob/main/DETAILED_REPORT.md))

*A showcase of the methodology. Turn Internet ON in Kaggle settings so the public data loads.*

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
plt.rcParams['figure.figsize'] = (9, 5); plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
print('ready')

## Load public NFL history

nflverse publishes weekly player stats as public Parquet files. We read the last several
completed seasons directly with pandas (no special package), then build **leakage-safe**
features: every 'recent form' number uses only weeks *before* the week in question, so nothing
ever peeks at the future. That discipline is what makes the backtest honest.

In [ ]:
POSITIONS = ('QB', 'RB', 'WR', 'TE')
NFLVERSE = ('https://github.com/nflverse/nflverse-data/releases/'
            'download/player_stats/player_stats_{yr}.parquet')

def completed_seasons(n=6):
    from datetime import date
    y = date.today().year if date.today().month >= 8 else date.today().year - 1
    return list(range(y - n, y))

def load_weekly(seasons):
    frames, used = [], []
    for yr in seasons:
        try: frames.append(pd.read_parquet(NFLVERSE.format(yr=yr))); used.append(yr)
        except Exception: pass
    df = pd.concat(frames, ignore_index=True)
    if 'season_type' in df.columns: df = df[df['season_type'] == 'REG']
    df = df[df['position'].isin(POSITIONS)].copy()
    df['ppr'] = pd.to_numeric(df.get('fantasy_points_ppr', 0), errors='coerce').fillna(0.0)
    print('loaded seasons', used, '|', f'{len(df):,} player-weeks')
    return df[['player_id','player_display_name','position','season','week','ppr']]

def prepare(df):
    df = df.sort_values(['player_id','season','week']).reset_index(drop=True)
    g = df.groupby(['player_id','season'])['ppr']
    df['roll3'] = g.transform(lambda s: s.rolling(3, min_periods=1).mean().shift(1))
    df['ttd']   = g.transform(lambda s: s.expanding().mean().shift(1))
    df['lw']    = g.shift(1)
    df['proj']  = 0.3*df['roll3'] + 0.7*df['ttd']   # recency-weighted season avg
    fut = pd.DataFrame({i: g.shift(-i) for i in (1,2,3)})
    df['fut3'] = fut.mean(axis=1); df['fut_n'] = fut.notna().sum(axis=1)
    return df

df = prepare(load_weekly(completed_seasons(6)))
df[['player_display_name','position','season','week','ppr','roll3','ttd','proj']].head()

## 1. Monte Carlo floor / ceiling

A single projected number hides risk: 12 points could be a safe 10–14 or a boom/bust 3–28.
We learn each position's week-to-week **performance-multiplier** distribution (weekly points ÷
season average) from history, then simulate thousands of outcomes around any projection to read
off a floor (P10), median (P50), and ceiling (P90).

In [ ]:
def tier(avg): return 'low' if avg < 9 else ('mid' if avg < 14 else 'high')
m = df.copy(); m['savg'] = m.groupby(['player_id','season'])['ppr'].transform('mean')
m = m[m['savg'] >= 6.0]; m['mult'] = m['ppr']/m['savg']; m['tier'] = m['savg'].apply(tier)
pools = {}
for pos in POSITIONS:
    pm = m[m['position']==pos]; pools[(pos,'all')] = pm['mult'].to_numpy()
    for t in ('low','mid','high'):
        arr = pm.loc[pm['tier']==t,'mult'].to_numpy()
        if len(arr) >= 200: pools[(pos,t)] = arr

def simulate(mu, pos, n=5000, seed=0):
    if not mu or mu <= 0: return None
    mults = pools.get((pos, tier(mu)))
    if mults is None or len(mults)==0: mults = pools.get((pos,'all'))
    sims = mu * np.random.default_rng(seed).choice(mults, size=n)
    f,med,c = np.percentile(sims, [10,50,90]); return dict(floor=round(f,1),median=round(med,1),ceiling=round(c,1),sims=sims)

for pos in POSITIONS:
    r = simulate(12.0, pos); print(f'{pos}: floor {r["floor"]:5.1f} | median {r["median"]:5.1f} | ceiling {r["ceiling"]:5.1f}')

In [ ]:
fig, ax = plt.subplots()
for pos in POSITIONS:
    ax.hist(simulate(12.0,pos)['sims'], bins=40, histtype='step', lw=2, density=True, label=pos)
ax.axvline(12, color='k', ls='--', alpha=0.6, label='projection (12)')
ax.set_title('Simulated weekly outcomes for a 12-point projection, by position')
ax.set_xlabel('fantasy points (PPR)'); ax.set_xlim(0,40); ax.legend(); plt.show()

## 2. Value over replacement

12 points at quarterback is not worth 12 at tight end, because startable tight ends are scarce.
So a player's value is measured against a **freely available (replacement-level)** player at the
same position. That makes players comparable across positions, which is what waiver and trade
decisions need.

In [ ]:
DEFAULT_RANKS = {'QB':14,'RB':29,'WR':29,'TE':14}   # last startable rank, 12-team
def replacement_levels(ranks):
    out = {}
    for pos, rank in ranks.items():
        pos_df = df[df['position']==pos]
        def at_rank(s):
            a = np.sort(s.to_numpy())[::-1]; return float(a[min(rank-1,len(a)-1)]) if len(a) else 0.0
        per_week = pos_df.groupby(['season','week'])['ppr'].apply(at_rank)
        out[pos] = round(float(per_week.mean()),1)
    return out
repl = replacement_levels(DEFAULT_RANKS)
print('Replacement level (weekly PPR):', repl)

cands = [('QB',12),('RB',12),('WR',12),('TE',12),('TE',9)]
pd.DataFrame([{'player':f'{p} proj {mu}','position':p,'proj':mu,'replacement':repl[p],
               'value_over_repl':round(mu-repl[p],1)} for p,mu in cands]).sort_values('value_over_repl',ascending=False)

## 3. "Heating up" thresholds

How big a recent-form jump actually counts as notable? We learn it per position: the 75th
percentile of historical positive jumps in recent-form-versus-season-average becomes the bar a
current player must clear to be flagged as heating up.

In [ ]:
h = df.copy(); h['delta'] = h['roll3'] - h['ttd']
notable = h[(h['roll3'] >= 4.0) & (h['delta'] > 0)]
thr = notable.groupby('position')['delta'].quantile(0.75).reindex(POSITIONS).round(2)
print('Notable-rise thresholds (PPG above own average):'); print(thr.to_string())
fig, axes = plt.subplots(2,2, figsize=(11,7), sharex=True)
for ax,pos in zip(axes.ravel(), POSITIONS):
    ax.hist(notable.loc[notable['position']==pos,'delta'], bins=40, color='#e8590c', alpha=.75)
    ax.axvline(thr[pos], color='k', ls='--', lw=2); ax.set_title(f'{pos} (bar +{thr[pos]:.2f})'); ax.set_xlim(0,12)
fig.suptitle('How big a recent-form rise counts as "heating up"'); plt.tight_layout(); plt.show()

## 4. Backtesting: does it actually work?

The most important part. We replay the recommendation logic across the loaded seasons, using
**only leakage-safe features** (nothing from the week being predicted), and grade it against
what really happened.

### Start/sit accuracy, and why confidence matters
For every pair of comparable players in a week, when the model projects one over the other by a
margin, did that player actually outscore the other? Accuracy climbs as the projected gap grows,
so the tool is right much more often when it's confident.

In [ ]:
def start_sit_accuracy(margin, pool_size=30):
    d = df.dropna(subset=['proj']).copy(); d = d[d['proj']>0]
    d['rk'] = d.groupby(['season','week','position'])['proj'].rank(ascending=False, method='first')
    d = d[d['rk'] <= pool_size]
    keys = ['season','week','position']; left = d[keys+['player_id','proj','ppr']]
    pairs = left.merge(left, on=keys, suffixes=('_a','_b'))
    pairs = pairs[pairs['player_id_a'] != pairs['player_id_b']]
    rec = pairs[pairs['proj_a'] - pairs['proj_b'] >= margin]
    return len(rec), (rec['ppr_a'] > rec['ppr_b']).mean()*100

rows = [(mg,)+start_sit_accuracy(mg) for mg in [2,4,6,8,10,14]]
res = pd.DataFrame(rows, columns=['min_gap','calls','accuracy_pct'])
print(res.to_string(index=False))
fig, ax = plt.subplots()
ax.bar(res['min_gap'].astype(str), res['accuracy_pct'], color='#1971c2')
ax.axhline(50, color='r', ls='--', label='coin flip (50%)')
ax.set_xlabel('only call it if projected gap ≥ (points)'); ax.set_ylabel('accuracy %')
ax.set_ylim(45,80); ax.set_title('Start/sit accuracy rises with confidence'); ax.legend()
for i,v in enumerate(res['accuracy_pct']): ax.text(i, v+0.5, f'{v:.0f}%', ha='center')
plt.show()

### Waiver value lift
Flag the best waiver-pool players (currently below replacement) using data through week W, then
measure the next three weeks. Flagged players become startable far more often than a random add.

In [ ]:
d = df.dropna(subset=['proj','roll3','ttd']).copy(); d = d[d['fut_n']>=1]
d['repl'] = d['position'].map(repl); pool = d[d['ttd'] < d['repl']].copy()
pool['value'] = (pool['proj'] - pool['repl']) + 0.5*(pool['roll3']-pool['ttd']).clip(lower=0)
pool['rk'] = pool.groupby(['season','week','position'])['value'].rank(ascending=False, method='first')
flagged = pool[pool['rk'] <= 3]
flag_hit = (flagged['fut3'] > flagged['repl']).mean()*100
base_hit = (pool['fut3'] > pool['repl']).mean()*100
print(f'Flagged became startable: {flag_hit:.0f}% vs base rate {base_hit:.0f}%  ->  {flag_hit/base_hit:.1f}x lift')
fig, ax = plt.subplots(figsize=(6,5))
ax.bar(['Random add\n(base rate)','Tool\'s flagged pick'], [base_hit, flag_hit], color=['#adb5bd','#2f9e44'])
ax.set_ylabel('% that became startable (next 3 weeks)'); ax.set_title(f'Waiver picks pan out {flag_hit/base_hit:.1f}x more often')
for i,v in enumerate([base_hit, flag_hit]): ax.text(i, v+0.5, f'{v:.0f}%', ha='center')
plt.show()

### Lineup capture
How much of the week's *achievable* points did the model's lineup capture, versus a naive
'chase last week's scorers' strategy?

In [ ]:
STARTERS = {'QB':1,'RB':2,'WR':2,'TE':1}
d = df.dropna(subset=['proj','ttd']).copy(); d = d[d['proj']>0]
mp=cp=xp=0.0
for (_s,_w,pos),grp in d.groupby(['season','week','position']):
    k = STARTERS.get(pos,1); pool2 = grp.sort_values('proj',ascending=False).head(30)
    if len(pool2)<k: continue
    best = pool2.sort_values('ppr',ascending=False).head(k)['ppr'].sum()
    if best<=0: continue
    mp += pool2.sort_values('proj',ascending=False).head(k)['ppr'].sum()
    cp += pool2.dropna(subset=['lw']).sort_values('lw',ascending=False).head(k)['ppr'].sum()
    xp += best
print(f'Model captured {mp/xp*100:.1f}% of optimal | chasing last week {cp/xp*100:.1f}%')

## 5. Are the ranges trustworthy? (Calibration)

A floor-to-ceiling range is only useful if you can trust it. A well-calibrated 80% range should
contain the real result about 80% of the time. We fit the range width on some seasons and test
it on a **held-out** season it never saw, the honest way to check.

In [ ]:
def tier_col(p): return np.where(p<9,'low',np.where(p<14,'mid','high'))
def pct_table(train, q):
    t = train.copy(); t['savg'] = t.groupby(['player_id','season'])['ppr'].transform('mean')
    t = t[t['savg']>=6.0]; t['mult'] = t['ppr']/t['savg']; t['tier'] = tier_col(t['savg'])
    rows=[]
    for pos in POSITIONS:
        pm = t[t['position']==pos]; allv = pm['mult'].to_numpy()
        if len(allv): rows.append((pos,'all',np.percentile(allv,q),np.percentile(allv,100-q)))
        for tr in ('low','mid','high'):
            a = pm.loc[pm['tier']==tr,'mult'].to_numpy()
            if len(a)>=200: rows.append((pos,tr,np.percentile(a,q),np.percentile(a,100-q)))
    return pd.DataFrame(rows, columns=['position','tier','p_lo','p_hi'])
def coverage(ev, tbl):
    d = ev.dropna(subset=['proj']).copy(); d = d[d['proj']>=4.0]; d['tier']=tier_col(d['proj'])
    d = d.merge(tbl[tbl.tier!='all'], on=['position','tier'], how='left')
    d = d.merge(tbl[tbl.tier=='all'].drop(columns='tier'), on='position', how='left', suffixes=('','_all'))
    d['p_lo']=d['p_lo'].fillna(d['p_lo_all']); d['p_hi']=d['p_hi'].fillna(d['p_hi_all']); d=d.dropna(subset=['p_lo','p_hi'])
    inside = (d['ppr']>=d['proj']*d['p_lo']) & (d['ppr']<=d['proj']*d['p_hi'])
    return inside.mean()*100
seasons = sorted(df['season'].unique()); train, test = df[df.season.isin(seasons[:-1])], df[df.season==seasons[-1]]
raw = coverage(test, pct_table(train, 10.0))
q = next((x/2 for x in range(20,1,-1) if coverage(train, pct_table(train,x/2))>=80), 10.0)
cal = coverage(test, pct_table(train, q))
print(f'Held-out {seasons[-1]}: raw 80% band covered {raw:.1f}% -> calibrated {cal:.1f}% (target 80)')
fig, ax = plt.subplots(figsize=(6,5))
ax.bar(['Raw band','Calibrated band'], [raw, cal], color=['#adb5bd','#2f9e44'])
ax.axhline(80, color='r', ls='--', label='target 80%'); ax.set_ylabel('% of outcomes inside the range')
ax.set_ylim(60,90); ax.set_title('Calibrating the prediction range (held-out season)'); ax.legend()
for i,v in enumerate([raw,cal]): ax.text(i, v+0.5, f'{v:.1f}%', ha='center')
plt.show()

## 6. Game script from live Vegas lines

Heavy favorites run the ball late (good for their running backs); heavy underdogs throw to catch
up (good for pass-catchers). ESPN's public scoreboard gives point spreads and over/unders, from
which we compute each team's **implied point total**. *(Live cell, output depends on the week.)*

In [ ]:
import re
SB='https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard'
def fetch_odds(week=None):
    try: data = requests.get(SB, params={'week':str(week)} if week else {}, timeout=30).json()
    except Exception as e: print('odds fetch failed', e); return {}
    out={}
    for ev in data.get('events',[]):
        for c in ev.get('competitions',[]):
            comps=c.get('competitors',[]); odds=c.get('odds') or []
            if len(comps)<2 or not odds: continue
            home=next((x for x in comps if x.get('homeAway')=='home'),None)
            away=next((x for x in comps if x.get('homeAway')=='away'),None)
            if not home or not away: continue
            ha=(home['team'].get('abbreviation') or '').upper(); aa=(away['team'].get('abbreviation') or '').upper()
            tot=odds[0].get('overUnder'); det=(odds[0].get('details') or '').strip()
            mo=re.match(r'^([A-Z]{2,4})\s+(-?\d+(?:\.\d+)?)$', det)
            if not mo or tot is None: continue
            fav=mo.group(1).upper(); line=float(mo.group(2)); tot=float(tot)
            hs = line if fav==ha else -line
            out[ha]={'spread':hs,'implied':round((tot-hs)/2,1)}; out[aa]={'spread':-hs,'implied':round((tot+hs)/2,1)}
    return out
odds = fetch_odds()
if odds:
    top = sorted(odds.items(), key=lambda kv: kv[1]['implied'], reverse=True)[:6]
    print('Highest implied team totals this slate (most expected scoring):')
    for t,v in top: print(f'  {t}: spread {v["spread"]:+}, implied {v["implied"]} pts')
else:
    print('No odds posted for the current slate yet.')

## Summary

On free, public data this notebook reproduces the tool's models and, more importantly, **proves**
they work: start/sit accuracy well above a coin flip (and higher when confident), waiver picks
that pan out more than twice as often as random, and prediction ranges calibrated on a season the
model never saw.

The live tool (private-league monitoring, Discord bot with /ask, /ranges and /trade, GitHub
Actions scheduling) and the full write-ups are in the repo:

### 🔗 [github.com/jamesonjdavis-maker/fantasy-football-monitor](https://github.com/jamesonjdavis-maker/fantasy-football-monitor)

**Honest scope:** the backtest validates the decision logic plus a reconstructed projection, not
ESPN's live projections (which aren't published for past seasons), so it likely understates the
live tool. Results are described as backtested, not guaranteed, and football is high-variance, so
no tool calls two closely-matched players reliably.